In [16]:
from torch.optim import AdamW, Optimizer
import torch.nn.functional as F

In [17]:
from config import SAVE_DATA_PATH
from data.load_data import get_dataloaders

train_loader, val_loader, test_loader = get_dataloaders(SAVE_DATA_PATH)

Loading dataset from disk:   0%|          | 0/18 [00:00<?, ?it/s]

In [18]:
from model.load_model import get_unires_model


model = get_unires_model()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_at

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[transformers] CLIPTextModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
vision_model.pre_layrnorm.bias                                 | UNEXPECTED |  | 
vision_model.embedding

In [19]:
from config import LEARNING_RATE


optimizer = AdamW(model.parameters(), LEARNING_RATE)

In [ ]:
import torch

from config import DICE_SMOOTH


def get_dice_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    # (B,224,224)
    probs = torch.sigmoid(logits)

    # flatten
    batch_size = logits.shape[0]
    # (B,50176)
    probs = probs.view(batch_size, -1)
    # (B,50176)
    targets = targets.view(batch_size, -1)

    # (B,)
    numerator = 2 * (probs * targets).sum(dim=1)
    # (B,)
    denominator = probs.sum(dim=1) + targets.sum(dim=1)
    # (B,)
    score = (numerator + DICE_SMOOTH) / (denominator + DICE_SMOOTH)
    loss = (1 - score).mean()

    return loss


def total_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    bce_loss = F.binary_cross_entropy_with_logits(logits, targets)
    dice_loss = get_dice_loss(logits, targets)
    return bce_loss + dice_loss

In [ ]:
from config import SEED


torch.manual_seed(SEED)
total_loss(torch.rand(16, 224, 224).to("cuda"), torch.rand(16, 224, 224).to("cuda"))

tensor(1.1806, device='cuda:0')

In [22]:
total_loss(
    torch.rand(16, 224, 224).to("cuda"),
    torch.randint(0, 2, (16, 224, 224)).float().to("cuda"),
)

tensor(1.1800, device='cuda:0')

In [23]:
type(optimizer)

torch.optim.adamw.AdamW

In [24]:
isinstance(optimizer, Optimizer)

True

In [44]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter()

In [26]:
isinstance(writer, SummaryWriter)

True

In [36]:
from typing import Tuple

from config import IOU_EPSILON


def calculate_miou_oiou(
    preds: torch.Tensor, targets: torch.Tensor
) -> Tuple[float, float]:
    # (B,224,224)
    preds = preds.bool()
    # (B,224,224)
    targets = targets.bool()

    # (B,)
    intersection = (preds & targets).sum(dim=(1, 2)).float()
    # (B,)
    union = (preds | targets).sum(dim=(1, 2)).float()

    # (B,)
    sample_ious = intersection / (union + IOU_EPSILON)
    miou = sample_ious.mean().item()

    total_intersection = intersection.sum().item()
    total_union = union.sum().item()
    oiou = total_intersection / (total_union + IOU_EPSILON)

    return miou, oiou

In [28]:
type((1, 2))

tuple

In [40]:
def log_metrics(
    log_tag: str,  # train/validation/test
    preds: torch.Tensor,
    targets: torch.Tensor,
    loss: float,
) -> None:
    accuracy = (targets == preds).float().mean().item()
    miou, oiou = calculate_miou_oiou(preds, targets)

    writer.add_scalars(
        log_tag,
        {
            "loss": loss,
            "accuracy": accuracy,
            "miou": miou,
            "oiou": oiou,
        },
    )

In [38]:
from config import LOG_INTERVAL
from model.unires import UniRes


def training_step(model: UniRes, batch, batch_idx: int) -> torch.Tensor:
    pixel_values = batch["pixel_values"]
    input_ids = batch["input_ids"]
    attention_mask = batch["attention_mask"]

    logits = model(pixel_values, input_ids, attention_mask)
    targets = batch["seg_masks"]

    loss = total_loss(logits, targets.float())

    if batch_idx % LOG_INTERVAL == 0:
        preds = logits > 0
        log_metrics("train", preds, targets, loss.item())

    return loss

In [45]:
log_metrics(
    "train", torch.randint(0, 2, (16, 224, 224)), torch.randint(0, 2, (16, 224, 224)), 0.3
)

In [46]:
len(train_loader)

440